github link: https://github.com/s234824dtu/compsocsci-assignment1

contributions:
all sections: shared between Aksel Mads Madsen (s234824) & Peter Møller Naur (s241554)

## Part 1: Ready Made vs Custom Made Data

1. Custom-made experimental data, such as that in the Centola study, is useful because it allows for proper experimental randomization, which makes causal analysis unambiguous, allowing you to be more certain that it's effects are real. However because the situation is artifical there is no way of establishing that the behavior in experimental conditions is the same as what happens in the real world. 
To overcome this you need to real-world observation data. However, such data, requires establishing using some more complex analytical tools as well as at least one convincingly exogenous shock (such as the weather used in the Nicolaides article), which is rarely easily available.
2. Because of these defects giving a straightforward interpretation of either study is difficult. To establish whether the experimental study succeeds in saying anything about spread of behavior in real-world social networks, you need to investigate in what ways the experimental setup differs from the part of the real world you want information about, and whether these differences affect the observed effects.
To interpret a study like the one by Nicolaides you need to be convinced that the claimed exogenous effect really is exogenous (in this case it's the weather, which convincingly exogenous), and study the analytical methods.

## Part 2: Find Researchers using the OpenAlex API

In [179]:
import requests
import pandas as pd 
import pyalex
import pickle
import math
import itertools
import tqdm
import joblib


In [162]:
organizers = ['Yue Zheng', 'Jacob Habinek', 'Martin Arvidsson', 'Andreas Jungherr', 'Erika Fille T. Legara', 'Johannes Fagerlind', 'Yukun Jiao', 'Xiaojing Jie', 'Juhi Kulshrestha', 'Angélica Sousa da Mata', 'Marc Keuschnigg', 'Marc Sparhuber', 'Kazuki Sakamoto', 'Peter Hedström', 'Kwabena Afriyie Owusu', 'Petter Holme', 'David Garcia', 'Lauren Fahey', 'Anastasia Menshikova', 'Yize Zhang', 'Laura Lotero', 'Elliot Ash', 'Sonia Yeh', 'Hendrik Erz', 'Vsevolod Suschevskiy', 'Thomas Haase', 'Denise Pumain', 'Zhao Feng', 'Yuan Liao', 'Josef Ginnerskov', 'José Bener', 'Yun-Tsz Tsai (Clara)', 'Mirta Galesic', 'Yang Yue', 'Helpis Saleh', 'Noé Vidal-Naquet', 'May Lim', 'Nina Tahmasebi']
with open("poster_authors.pkl", "rb") as f:
    presenters = pickle.loads(f.read())
researchers = list(set(organizers).union(set(presenters)))
len(researchers)

948

In [165]:
with open("api_key", "r") as f:
    pyalex.config.api_key = str(f.read())

alex_info = []

for researcher_name in tqdm.tqdm(researchers):
    alex_info.append(pyalex.Authors().search(researcher_name).get())

100%|██████████| 948/948 [14:18<00:00,  1.10it/s]


In [214]:
def argmax(l):
    max_i = 0
    max_e = -math.inf
    for i, e in enumerate(l):
        if e > max_e:
            max_e = e
            max_i = i
    return max_i
        
def disambig(i):
    return alex_info[i][argmax([result["relevance_score"] for result in alex_info[i]])]

disambig_alex_info = [disambig(i) for i in range(len(alex_info)) if len(alex_info[i]) > 0]
len(disambig_alex_info)

906

In [215]:
def path_access(o, path):
    if len(path) == 0:
        return o
    else:
        next = o[path[0]]
        if next == None:
            return None
        return path_access(next, path[1:])

d = {}
for attr in [["id"], ["display_name"], ["works_api_url"], ["summary_stats", "h_index"], ["works_count"], ["last_known_institutions", 0, "country_code"]]:
    l = []
    for researcher_alex_info in disambig_alex_info:
        l.append(path_access(researcher_alex_info, attr))
    d[".".join((str(component) for component in attr))] = l

df = pd.DataFrame.from_dict(d)
df 

,id,display_name,works_api_url,summary_stats.h_index,works_count,last_known_institutions.0.country_code
0,https://openalex.org/A5017981514,Kohei Takeda,https://api.openalex.org/works?filter=author.i...,23,169,SG
1,https://openalex.org/A5051891666,Florian Keusch,https://api.openalex.org/works?filter=author.i...,25,122,DE
2,https://openalex.org/A5032880777,Michaël Maes,https://api.openalex.org/works?filter=author.i...,141,1385,US
3,https://openalex.org/A5053982410,Fábio Sartori,https://api.openalex.org/works?filter=author.i...,14,129,IT
4,https://openalex.org/A5084087666,Wanting Wang,https://api.openalex.org/works?filter=author.i...,22,152,CN
...,...,...,...,...,...,...
901,https://openalex.org/A5022729086,Bijin Joseph,https://api.openalex.org/works?filter=author.i...,2,5,IN
902,https://openalex.org/A5026595284,Xiaoyuan Yi,https://api.openalex.org/works?filter=author.i...,4,14,CN
903,https://openalex.org/A5012128678,Max R. P. Grossmann,https://api.openalex.org/works?filter=author.i...,2,9,DE
904,https://openalex.org/A5057457548,Gang Sun,https://api.openalex.org/works?filter=author.i...,81,546,CN


In [169]:
df.to_pickle("researchers.pickle")

We found that when searching for names in the openalex database, there were multiple results for each name most of the time, meaning we had to disambiguate between similarly named researchers. 
Initially we attempted to map the concepts and topics noted for each author identification candidate to the quantitative social science field in general or to the subjects of the poster. 
However, the classification of concepts in the OpenAlex database was not precise enough, and most authors could not be disambiguated in this way. 
We had to settle for choosing the author candidate with the highest relevance score from the OpenAlex search tool, which does increase the chances that we are actually analyzing data about the wrong researchers.

## Part 3: Collect Research Articles

The IC2S2 papers dataset contains 27337 different works authored by 42368 different authors.

To make the code more efficient we implemented all the suggestions made in the problem description. 
In particular we
1. Used api-side filtering for the citation count, author count and concept.
2. Made requests for works from 25 authors per request.
3. Increased the pagination from 25 to 200 work received per response.
4. Ran 8 of the 25-author requests in parallel. 

Which put us at half at minute for the full download.

All the filters serve the purpose of narrowing the work couunt down the most likely papers to be relevant to the conference. The citation count filter low-impact papers, as does the paper count per author minimum limit. An author that authors more than 5000 papers can't be considered relevant to each ones, so we don't want to follow that link. The concepts filter attempt to remove the works from authors in other subjects.

Subfields with a higher threshold for publishing a paper will be underrepresented as a result of the minimum paper per author filter.
In addition we previously saw that the coarse-grained concepts in OpenAlex might be insufficient to identify works within a field. In particular we note that many of the works in the resulting dataset aren't directly relevant.

In [170]:
social_concepts = ["Sociology", "Psychology", "Economics", '"Political science"'] # pyablex doesn't automatically quote
quant_concepts = ["Mathematics", "Physics", '"Computer science"']
def get_concepts_id(concept_names):
    return [pyalex.Concepts().filter(display_name=name, level=0).get()[0]["id"] for name in concept_names]

social_concept_ids = get_concepts_id(social_concepts)
quant_concept_ids = get_concepts_id(quant_concepts)


/tmp/ipykernel_14920/1165536202.py:4: DeprecationWarning: Concepts is deprecated by OpenAlex and replaced by topics.
  return [pyalex.Concepts().filter(display_name=name, level=0).get()[0]["id"] for name in concept_names]


In [181]:
def select_author_id(work):
    return {
        "id": work["id"],
        "publication_year": work["publication_year"],
        "cited_by_count": work["cited_by_count"],
        "author_ids": [authorship["author"]["id"] for authorship in work["authorships"]],
        "title": work["title"],
        "abstract_inverted_index": work["abstract_inverted_index"]
    } 

def fetch_author_list(author_info_l):
    ids = [author_info["id"] for author_info in author_info_l]
    fields = ["id", "publication_year", "cited_by_count", "authorships", "title", "abstract_inverted_index"]
    works = pyalex.Works().filter(authorships={"author": {"id": "|".join(ids)}}, authors_count="<10", cited_by_count=">10", concepts={"id": ["|".join(social_concept_ids), "|".join(quant_concept_ids)]}).select(fields).paginate(per_page=200)
    return [select_author_id(work) for work in itertools.chain(*works)]


In [197]:
from joblib import delayed
relavant_authors = [author for author in disambig_alex_info
                    if author["works_count"] > 5 and author["works_count"] < 5000]

block_size = 25
r = joblib.Parallel(n_jobs=8)(delayed(lambda i: fetch_author_list(relavant_authors[i*block_size:(i+1)*block_size]))(i) for i in range(math.ceil(len(relavant_authors)/block_size)))

In [ ]:
all_works = [work for job in r for work in job]
all_works_unique = list({work["id"]: work for work in all_works}.values())

len(all_works_unique)

25851

In [213]:
all_unique_authors_ids = list({author_id for work in all_works_unique
                         for author_id in work["author_ids"]})
len(all_unique_authors_ids)

42368

In [211]:
d = {}
for attr in ["id", "publication_year", "cited_by_count", "author_ids"]:
    l = []
    for work in all_works:
        l.append(work[attr])
    d[attr] = l

df1 = pd.DataFrame.from_dict(d)
df1.to_pickle("works.pickle")
df1

,id,publication_year,cited_by_count,author_ids
0,https://openalex.org/W2108598243,2009.0,60261,"[https://openalex.org/A5101542158, https://ope..."
1,https://openalex.org/W4239072543,2009.0,2917,"[https://openalex.org/A5101480290, https://ope..."
2,https://openalex.org/W2250384498,2016.0,1500,"[https://openalex.org/A5015683164, https://ope..."
3,https://openalex.org/W2016674662,2009.0,1407,"[https://openalex.org/A5042890616, https://ope..."
4,https://openalex.org/W1497385253,2004.0,1300,"[https://openalex.org/A5103339501, https://ope..."
...,...,...,...,...
27332,https://openalex.org/W2892182042,2019.0,12,"[https://openalex.org/A5055710645, https://ope..."
27333,https://openalex.org/W431568915,2015.0,12,"[https://openalex.org/A5086827245, https://ope..."
27334,https://openalex.org/W1542860446,2000.0,11,"[https://openalex.org/A5091405758, https://ope..."
27335,https://openalex.org/W2950292336,2016.0,11,"[https://openalex.org/A5101577090, https://ope..."


In [212]:
d = {}
for attr in ["id", "title", "abstract_inverted_index"]:
    l = []
    for work in all_works:
        l.append(work[attr])
    d[attr] = l

df2 = pd.DataFrame.from_dict(d)
df2.to_pickle("works.pickle")
df2

,id,title,abstract_inverted_index
0,https://openalex.org/W2108598243,ImageNet: A large-scale hierarchical image dat...,"{'The': [0], 'explosion': [1], 'of': [2, 56, 6..."
1,https://openalex.org/W4239072543,ImageNet: A large-scale hierarchical image dat...,"{'The': [0], 'explosion': [1], 'of': [2, 56, 6..."
2,https://openalex.org/W2250384498,YFCC100M,"{'This': [0], 'publicly': [1], 'available': [2..."
3,https://openalex.org/W2016674662,Multiscale mobility networks and the spatial s...,"{'Among': [0], 'the': [1, 8, 22, 29, 37, 49, 6..."
4,https://openalex.org/W1497385253,Activity Recognition in the Home Using Simple ...,None
...,...,...,...
27332,https://openalex.org/W2892182042,Simplicity Creates Inequity: Implications for ...,"{'Algorithms': [0], 'are': [1, 49, 219], 'incr..."
27333,https://openalex.org/W431568915,Dissecting the Spirit of Gezi: Influence vs. S...,None
27334,https://openalex.org/W1542860446,Algorithms for Constructing Comparative Maps,None
27335,https://openalex.org/W2950292336,Image-embodied Knowledge Representation Learning,"{'Entity': [0], 'images': [1, 61], 'could': [2..."
